In [31]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
import numpy as np

class ImageDFDataset(Dataset):
    def __init__(self, df, label_to_idx, transform=None):
        self.df = df.reset_index(drop=True)
        self.label_to_idx = label_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = '../data' + self.df.loc[idx, "image_path"]
        label_str = self.df.loc[idx, "label"]
        label = self.label_to_idx[label_str]

        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, label


def build_label_mapping(df_train):
    classes = sorted(df_train["label"].unique())
    label_to_idx = {c: i for i, c in enumerate(classes)}
    idx_to_label = {i: c for c, i in label_to_idx.items()}
    return label_to_idx, idx_to_label


train_transform = transforms.Compose([
    transforms.RandomResizedCrop(128, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.1),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.5),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(144),
    transforms.CenterCrop(128),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

def make_dataloaders(df_train, df_val, batch_size=32):

    label_to_idx, idx_to_label = build_label_mapping(df_train)
    num_classes = len(label_to_idx)

    train_dataset = ImageDFDataset(df_train, label_to_idx, transform=train_transform)
    val_dataset   = ImageDFDataset(df_val, label_to_idx, transform=val_transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, num_classes, label_to_idx, idx_to_label

def build_resnet18(num_classes):
    model = models.resnet18(weights=None)   # NOT pretrained

    model.fc = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(512, num_classes)
    )

    return model


def train_model(model, train_loader, val_loader, epochs=100, lr=1e-3):

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    for epoch in range(epochs):
        model.train()
        running_loss = 0


        for imgs, labels in train_loader:
            optimizer.zero_grad()
            imgs, labels_a, labels_b, lam = mixup(imgs, labels)
            imgs, labels_a, labels_b  = imgs.to(device), labels_a.to(device), labels_b.to(device)
            outputs = model(imgs)
            loss = lam * criterion(outputs, labels_a) + (1 - lam) * criterion(outputs, labels_b)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        val_loss = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        print(f"Epoch {epoch+1}/{epochs} "
              f"| Train Loss: {running_loss/len(train_loader):.4f} "
              f"| Val Loss: {val_loss:.4f}")

@torch.inference_mode()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        total_loss += loss.item()

    return total_loss / len(loader)

def mixup(x, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size()[0]
    index = torch.randperm(batch_size)

    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

In [33]:
model = build_resnet18(200)

In [ ]:
from sklearn.model_selection import train_test_split
# ============================================
# 8. Example usage
# ============================================
df = pd.read_csv("../data/train_images.csv")


In [37]:
df['image_path'] = df['image_path'].str.replace('train_images', 'train_images_rembg')
df['image_path'] = df['image_path'].str.replace('jpg', 'png')

In [39]:
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df['label'])

train_loader, val_loader, num_classes, label_to_idx, idx_to_label = make_dataloaders(train_df, val_df)

In [43]:
train_model(model, train_loader, val_loader, epochs=120, lr=1e-3)

Epoch 1/120 | Train Loss: 5.1930 | Val Loss: 5.0283
Epoch 2/120 | Train Loss: 5.0620 | Val Loss: 5.0926
Epoch 3/120 | Train Loss: 4.9914 | Val Loss: 5.2849
Epoch 4/120 | Train Loss: 4.9841 | Val Loss: 4.9827
Epoch 5/120 | Train Loss: 4.9438 | Val Loss: 5.2031
Epoch 6/120 | Train Loss: 4.9055 | Val Loss: 4.6653
Epoch 7/120 | Train Loss: 4.8656 | Val Loss: 5.3118
Epoch 8/120 | Train Loss: 4.8568 | Val Loss: 4.6484
Epoch 9/120 | Train Loss: 4.8492 | Val Loss: 4.7021
Epoch 10/120 | Train Loss: 4.7909 | Val Loss: 4.6924
Epoch 11/120 | Train Loss: 4.7362 | Val Loss: 4.7343
Epoch 12/120 | Train Loss: 4.7152 | Val Loss: 4.5683
Epoch 13/120 | Train Loss: 4.6961 | Val Loss: 4.6008
Epoch 14/120 | Train Loss: 4.6377 | Val Loss: 4.3276
Epoch 15/120 | Train Loss: 4.6954 | Val Loss: 4.6489
Epoch 16/120 | Train Loss: 4.5826 | Val Loss: 4.9691
Epoch 17/120 | Train Loss: 4.5753 | Val Loss: 4.6924
Epoch 18/120 | Train Loss: 4.5259 | Val Loss: 4.5505
Epoch 19/120 | Train Loss: 4.5035 | Val Loss: 4.1474
Ep

KeyboardInterrupt: 

In [44]:
test_df = pd.read_csv("../data/test_images_path.csv")
test_df['image_path'] = test_df['image_path'].str.replace('test_images', 'test_images_rembg')
test_df['image_path'] = test_df['image_path'].str.replace('jpg', 'png')
test_dataset   = ImageDFDataset(test_df, label_to_idx, transform=val_transform)
test_loader   = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [45]:
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [46]:
all_ids = test_df["id"].tolist()
all_preds = []

In [47]:
with torch.no_grad():
    for inputs, _ in test_loader:
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.numpy())

In [48]:
predicted_labels = [idx_to_label[i] for i in all_preds]

In [49]:
output_df = pd.DataFrame({
    "id": all_ids,
    "label": predicted_labels
})

output_df.to_csv("test_predictions.csv", index=False)
print("Saved test_predictions.csv!")

Saved test_predictions.csv!


In [50]:
from sklearn.metrics import accuracy_score
val_preds = []
val_labels = []
model.eval()

with torch.no_grad():
    for inputs, labels in val_loader:
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)

        val_preds.extend(predicted.numpy())
        val_labels.extend(labels.numpy())

accuracy = accuracy_score(val_labels, val_preds)

In [51]:
accuracy

0.4529262086513995